In [ ]:
import sys
import os
from pathlib import Path

# current_dir=str(Path(os.getcwd()).parent.parent.parent)
# print(f"Current dir: {current_dir}")
# sys.path.insert(0, current_dir)

if __name__ == "__main__":
    root_path = str(Path(os.getcwd()).parents[2])
    print(f"Root path: {root_path}")
    sys.path.insert(0, root_path)
    if __package__ is None:
        __package__ = "comfyui-image-scorer.external_modules.step03training.text_data"
    print(f"Package: {__package__}")

In [ ]:
from shared.paths import text_data_file, scores_file
from shared.io import load_single_jsonl
from tqdm import tqdm
from typing import List, Dict, Any, Tuple

text_data = load_single_jsonl(text_data_file)
scores = load_single_jsonl(scores_file)
print(f"text data: {len(text_data)} lines")
print(f"scores: {len(scores)} lines")

lora_data: Dict[str, List[Tuple[float, float]]] = {}
positive_prompt: Dict[str, List[Tuple[float, float]]] = {}
negative_prompt: Dict[str, List[Tuple[float, float]]] = {}
discrete_data: Dict[str, Dict[str | int, List[float]]] = {}
continuous_data: Dict[str, List[Tuple[float, float]]] = {}

with tqdm(total=len(text_data), desc="Processing text data", unit="line") as pbar:
    for i in range(len(text_data)):
        current_line: Dict[str, Any] = text_data[i]
        current_score: float = scores[i]

        current_lora_line: List[Tuple[float, float]] = (
            list(lora_data[current_line["lora"]])
            if current_line["lora"] in lora_data
            else []
        )
        current_lora_line.append((current_line["lora_weight"], current_score))
        lora_data[current_line["lora"]] = current_lora_line
        del current_line["lora"], current_line["lora_weight"]

        current_positive = current_line["positive_prompt"]
        for prompt, weight in current_positive:
            if prompt not in positive_prompt:
                positive_prompt[prompt] = []
            positive_prompt[prompt].append((weight, current_score))

        current_negative = current_line["negative_prompt"]
        for prompt, weight in current_negative:
            if prompt not in negative_prompt:
                negative_prompt[prompt] = []
            negative_prompt[prompt].append((weight, current_score))

        del current_line["negative_prompt"], current_line["positive_prompt"]

        for key, value in list(current_line.items()):
            if type(value) in [str, int]:
                if key not in discrete_data:
                    discrete_data[key] = {}
                current_column = discrete_data[key]
                if value not in current_column:
                    current_column[value] = []
                current_column[value].append(current_score)
                discrete_data[key] = current_column
            elif type(value) is float:
                if key not in continuous_data:
                    continuous_data[key] = []
                continuous_data[key].append((value, current_score))
            else:
                raise ValueError(f" {key} type not valid: {type(value)}")

        pbar.update(1)
print(f"Final data: {len(text_data)} lines")

Process data

In [ ]:
from shared.training.plot import PlotManager


PlotManager.plot_individual_metrics((continuous_data),cols=3,bins=15)
#PlotManager.plot_continuous_analysis(lora_data,"Lora", "weight","score")
#PlotManager.plot_aggregate_summary(lora_data,"Lora","score",top_percent=1, limit=10, ascending=False)
#PlotManager.plot_continuous_analysis(continuous_data," ", "value","score")
#PlotManager.plot_aggregate_summary(continuous_data,"Continuous data","score",top_percent=0.9, limit=10, ascending=False)
PlotManager.plot_aggregate_summary(positive_prompt,"Positive prompts","score",top_percent=0.8, limit=20,ascending=True)
#PlotManager.plot_discrete_analysis(discrete_data," ", "value","score")
PlotManager.plot_discrete_object_analysis(discrete_data)